In [ ]:
"""
UHI Transect Data Pull — 395 Riverside Dr corridor
Broadway side → West End Ave → Riverside Drive → Riverside Park
Week of April 7–14, 2026

Data sources:
  1. Open-Meteo (free, no key) — hourly air temp + humidity at 4 transect points
  2. Weather Underground PWS API (needs free API key) — nearby personal stations
  3. Output: CSV per point + combined comparison CSV

Install deps:
  pip install requests pandas openmeteo-requests retry-requests

Usage:
  python uhi_transect_395riverside.py
  python uhi_transect_395riverside.py --wu-key YOUR_KEY_HERE
"""

import argparse
import requests
import pandas as pd
from datetime import datetime, timezone

# ── Transect anchor points (W 111th St corridor) ─────────────────────────────
# Moving west from Broadway to Riverside Park
TRANSECT_POINTS = {
    "Broadway_111th":       {"lat": 40.8053, "lon": -73.9659, "desc": "Broadway & W111 (dense commercial)"},
    "WestEnd_111th":        {"lat": 40.8056, "lon": -73.9701, "desc": "West End Ave & W111 (mid-density residential)"},
    "RiversideDr_395":      {"lat": 40.8059, "lon": -73.9732, "desc": "395 Riverside Drive (your building)"},
    "RiversidePark_edge":   {"lat": 40.8061, "lon": -73.9755, "desc": "Riverside Park edge (tree canopy / green)"},
}

# Date range: this week
START_DATE = "2026-04-07"
END_DATE   = "2026-04-14"


# ── 1. Open-Meteo: free hourly data, no API key ───────────────────────────────

def fetch_openmeteo(name, lat, lon):
    """Pull hourly temp + humidity for the week from Open-Meteo (ERA5-land)."""
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude":              lat,
        "longitude":             lon,
        "start_date":            START_DATE,
        "end_date":              END_DATE,
        "hourly":                "temperature_2m,relativehumidity_2m,apparent_temperature",
        "temperature_unit":      "fahrenheit",
        "timezone":              "America/New_York",
    }
    r = requests.get(url, params=params, timeout=20)
    r.raise_for_status()
    data = r.json()["hourly"]
    df = pd.DataFrame({
        "datetime":    pd.to_datetime(data["time"]),
        "temp_f":      data["temperature_2m"],
        "humidity_pct":data["relativehumidity_2m"],
        "feels_like_f":data["apparent_temperature"],
        "location":    name,
    })
    return df


def pull_all_openmeteo():
    frames = []
    for name, pt in TRANSECT_POINTS.items():
        print(f"  Fetching Open-Meteo for {name}...")
        df = fetch_openmeteo(name, pt["lat"], pt["lon"])
        frames.append(df)
    combined = pd.concat(frames, ignore_index=True)
    return combined


# ── 2. Weather Underground PWS: nearby personal stations ─────────────────────

def fetch_wunderground_nearby(lat, lon, radius_km=1, api_key=None):
    """
    Find PWS stations within radius_km of a point and pull their 7-day history.
    Requires a free Weather Underground API key from:
    https://www.wunderground.com/member/api-keys
    """
    if not api_key:
        print("  [WU] No API key provided — skipping PWS pull.")
        print("       Get a free key at https://www.wunderground.com/member/api-keys")
        return None

    # Step 1: find nearby stations
    search_url = "https://api.weather.com/v3/location/near"
    search_params = {
        "geocode":  f"{lat},{lon}",
        "product":  "pws",
        "format":   "json",
        "apiKey":   api_key,
    }
    r = requests.get(search_url, params=search_params, timeout=15)
    if r.status_code != 200:
        print(f"  [WU] Station search failed: {r.status_code}")
        return None

    stations = r.json().get("location", {}).get("stationName", [])
    station_ids = r.json().get("location", {}).get("stationIdentifier", [])
    lons = r.json().get("location", {}).get("longitude", [])
    lats = r.json().get("location", {}).get("latitude", [])

    print(f"  [WU] Found {len(station_ids)} nearby stations")

    frames = []
    for sid, sname, slat, slon in zip(station_ids, stations, lats, lons):
        print(f"    → {sid} ({sname}) at {slat:.4f},{slon:.4f}")

        obs_url = "https://api.weather.com/v2/pws/observations/hourly/7day"
        obs_params = {
            "stationId": sid,
            "format":    "json",
            "units":     "e",          # imperial
            "apiKey":    api_key,
            "numericPrecision": "decimal",
        }
        ro = requests.get(obs_url, params=obs_params, timeout=15)
        if ro.status_code != 200:
            print(f"      [WU] Obs fetch failed for {sid}: {ro.status_code}")
            continue

        obs_list = ro.json().get("observations", [])
        rows = []
        for o in obs_list:
            rows.append({
                "datetime":     pd.to_datetime(o["obsTimeLocal"]),
                "station_id":   sid,
                "station_name": sname,
                "station_lat":  slat,
                "station_lon":  slon,
                "temp_f":       o.get("imperial", {}).get("temp"),
                "humidity_pct": o.get("humidity"),
                "wind_mph":     o.get("imperial", {}).get("windSpeed"),
                "precip_in":    o.get("imperial", {}).get("precipTotal"),
            })
        if rows:
            frames.append(pd.DataFrame(rows))

    return pd.concat(frames, ignore_index=True) if frames else None


# ── 3. Compute transect differential ─────────────────────────────────────────

def compute_differential(df):
    """
    Pivot the Open-Meteo data to show hourly ΔT between Broadway and Riverside Park.
    This is the core UHI signal.
    """
    pivot = df.pivot_table(
        index="datetime",
        columns="location",
        values="temp_f",
        aggfunc="mean",
    )

    if "Broadway_111th" in pivot.columns and "RiversidePark_edge" in pivot.columns:
        pivot["delta_broadway_minus_park"] = (
            pivot["Broadway_111th"] - pivot["RiversidePark_edge"]
        )
        pivot["delta_building_minus_park"] = (
            pivot["RiversideDr_395"] - pivot["RiversidePark_edge"]
        )

    return pivot.reset_index()


# ── 4. Summary statistics ─────────────────────────────────────────────────────

def print_summary(df):
    print("\n" + "="*60)
    print("TRANSECT TEMPERATURE SUMMARY  (Apr 7–14, 2026)")
    print("="*60)

    for loc, group in df.groupby("location"):
        desc = TRANSECT_POINTS.get(loc, {}).get("desc", loc)
        print(f"\n{desc}")
        print(f"  Mean temp:   {group['temp_f'].mean():.1f}°F")
        print(f"  Max temp:    {group['temp_f'].max():.1f}°F")
        print(f"  Min temp:    {group['temp_f'].min():.1f}°F")
        print(f"  Mean RH:     {group['humidity_pct'].mean():.0f}%")

    diff_df = compute_differential(df)
    if "delta_broadway_minus_park" in diff_df.columns:
        print("\n── UHI Signal ──────────────────────────────────────")
        print(f"  Mean Broadway − Park ΔT:    {diff_df['delta_broadway_minus_park'].mean():.2f}°F")
        print(f"  Peak daytime ΔT (10–16h):   ", end="")
        daytime = diff_df[diff_df["datetime"].dt.hour.between(10, 16)]
        print(f"{daytime['delta_broadway_minus_park'].mean():.2f}°F")
        print(f"  Mean 395 Riverside − Park:  {diff_df['delta_building_minus_park'].mean():.2f}°F")


# ── 5. Export ─────────────────────────────────────────────────────────────────

def export(df, diff_df, wu_df=None):
    df.to_csv("transect_hourly_openmeteo.csv", index=False)
    diff_df.to_csv("transect_differential.csv", index=False)
    if wu_df is not None:
        wu_df.to_csv("transect_pws_wunderground.csv", index=False)
    print("\nFiles saved:")
    print("  transect_hourly_openmeteo.csv   — all 4 points, hourly")
    print("  transect_differential.csv       — ΔT between Broadway & park")
    if wu_df is not None:
        print("  transect_pws_wunderground.csv   — nearby PWS station readings")


# ── Main ──────────────────────────────────────────────────────────────────────

def main():
    parser = argparse.ArgumentParser(description="Pull UHI transect data near 395 Riverside Dr")
    parser.add_argument("--wu-key", default=None, help="Weather Underground API key (optional)")
    args = parser.parse_args()

    print("Pulling Open-Meteo hourly data for 4 transect points...")
    df = pull_all_openmeteo()

    print("\nComputing differentials...")
    diff_df = compute_differential(df)

    print("\nQuerying Weather Underground for nearby PWS stations...")
    # Use 395 Riverside Dr as the center point for PWS search
    center = TRANSECT_POINTS["RiversideDr_395"]
    wu_df = fetch_wunderground_nearby(center["lat"], center["lon"], api_key=args.wu_key)

    print_summary(df)
    export(df, diff_df, wu_df)


if __name__ == "__main__":
    main()